# UnifyWeaver における高度な再帰パターン

このノートブックでは、UnifyWeaver が検出および最適化できる4つの主要な再帰パターンを実演します:

1. **末尾再帰（Tail Recursion）** - アキュムレータを用いた反復ループ
2. **線形再帰（Linear Recursion）** - メモ化を伴う単一の再帰呼び出し
3. **木再帰（Tree Recursion）** - 構造の構成要素に対する複数の再帰呼び出し
4. **相互再帰（Mutual Recursion）** - 複数の述語が循環して互いを呼び出し合う構造

## 学習目標

- さまざまな再帰パターンを理解する
- UnifyWeaver が各パターンをどのように検出・最適化するかを確認する
- パフォーマンス特性を比較する
- 各パターンをいつ使用すべきかを学ぶ

## セットアップ

UnifyWeaver 環境を初期化します。

In [ ]:
% 初期化をロード
['../init'].

% 必要なモジュールをロード
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## パターン 1: 末尾再帰（Tail Recursion）

末尾再帰はアキュムレータ（累積変数）を使用して中間結果を保持し、再帰呼び出しが関数内の**最後のアクション**となります。

### 例: リスト内の要素数をカウントする

In [ ]:
% 末尾再帰の count_items を定義
:- dynamic count_items/3.

% 基底ケース: 空リスト、アキュムレータを返す
count_items([], Acc, Acc).

% 再帰ケース: アキュムレータをインクリメントし、リストの末尾に対して再帰
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← 末尾位置！

### Prolog でのテスト

In [ ]:
% テスト: [a,b,c,d,e] の要素数をカウント
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### パターン検出の確認

In [ ]:
% 末尾再帰として検出されたか確認
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Bash へのコンパイル

In [ ]:
% コンパイルして保存
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### 生成された Bash のテスト

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "[a,b,c,d,e] の要素をカウント中:"
count_items "[a,b,c,d,e]" 0 ""

## パターン 2: 線形再帰（Linear Recursion）

線形再帰は節ごとに**ちょうど1つ**の再帰呼び出しを持ち、再帰呼び出しが戻った後に計算が行われます。

### 例: 階乗（Factorial）

In [ ]:
% 階乗を定義
:- dynamic factorial/2.

% 基底ケース
factorial(0, 1).

% 再帰ケース: ちょうど1回の再帰呼び出し
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← 1回の再帰呼び出し
    F is N * F1.        % ← 呼び出し後の計算

### Prolog でのテスト

In [ ]:
% テスト: 5の階乗
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### パターン検出の確認

In [ ]:
% 線形再帰として検出されたか確認
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Bash へのコンパイル

In [ ]:
% コンパイルして保存
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % 関数定義のみを保持（Brush は source されたスクリプトを直接実行として処理するため）
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### 生成された Bash のテスト

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "5の階乗:"
factorial 5 ""
echo ""
echo "10の階乗:"
factorial 10 ""

## パターン 3: 木再帰（Tree Recursion）

木再帰は、構造の異なる部分を処理するために**複数**の再帰呼び出しを行います。

### 例: 木構造の総和（Tree Sum）

In [ ]:
% 二分木用の tree_sum を定義
% 木構造の形式: [値, 左部分木, 右部分木] または []
:- dynamic tree_sum/2.

% 基底ケース: 空の木の合計は0
tree_sum([], 0).

% 再帰ケース: 合計 = 値 + 左の合計 + 右の合計
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← 1回目の再帰呼び出し
    tree_sum(R, RS),   % ← 2回目の再帰呼び出し
    Sum is V + LS + RS.

### Prolog でのテスト

In [ ]:
% テスト: [5, [3, [1, [], []], []], [2, [], []]] の tree_sum
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Bash へのコンパイル

In [ ]:
% コンパイルして保存
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### 生成された Bash のテスト

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "[5,[3,[1,[],[]],[]],[2,[],[]]] の木の合計:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## パターン 4: 相互再帰（Mutual Recursion）

相互再帰は、2つ以上の述語が循環して互いを呼び出すときに発生します。

### 例: 偶数（Even）と奇数（Odd）

In [ ]:
% 相互再帰の is_even と is_odd を定義
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even の基底ケース
is_even(0).

% is_even の再帰: N-1 が奇数なら N は偶数
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← is_odd を呼び出し

% is_odd の基底ケース
is_odd(1).

% is_odd の再帰: N-1 が偶数なら N は奇数
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← is_even を呼び出し

### Prolog でのテスト

In [ ]:
% 偶数/奇数のテスト
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### 相互再帰のチェック

In [ ]:
% コールグラフを構築して SCC を検出
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Bash へのコンパイル

In [ ]:
% 相互再帰グループをコンパイル
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### 生成された Bash のテスト

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "is_even と is_odd をテスト中:"
is_even 0 >/dev/null && echo "✓ 0 は偶数"
is_even 4 >/dev/null && echo "✓ 4 は偶数"
is_odd 3 >/dev/null && echo "✓ 3 は奇数"
is_odd 7 >/dev/null && echo "✓ 7 は奇数"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 は偶数ではない"

## パターンの比較

各パターンの特徴を比較してみましょう:

| パターン | 再帰呼び出し | 最適化手法 | 空間計算量 | 主な用途 |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **末尾再帰** | 1回（末尾位置） | 反復ループ | O(1) | アキュムレータ、線形スキャン |
| **線形再帰** | 1回（任意の位置） | 畳み込み（Fold） + メモ化 | O(n) メモテーブル | フィボナッチ、階乗 |
| **木再帰** | 2回以上（構造の各部） | 構造分解 | O(depth) スタック | 木/グラフ操作 |
| **相互再帰** | 1回以上（述語間） | 共有メモ化 | O(n) 共有テーブル | 偶数/奇数、相互定義 |

## パターン検出順序

UnifyWeaver は以下の優先順位でパターン検出を試みます:

1. **末尾再帰**（最も効率的）
2. **線形再帰**（禁止されていない限り）
3. **木再帰**（構造的）
4. **相互再帰**（強連結成分 (SCC) 検出）
5. **基本再帰**（フォールバック）

`forbid_linear_recursion/1` を使用して検出挙動を制御できます。

## 演習問題: チャレンジしてみよう！

以下の述語を定義してコンパイルしてみましょう:

### 1. 末尾再帰による総和（Sum）
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. 線形再帰によるフィボナッチ数（Fibonacci）
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. 木の高さ（Tree Height）
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% ここにコードを記述！


## まとめ

このノートブックで学んだこと:

✅ UnifyWeaver における4つの主要な再帰パターン

✅ Prolog で各パターンを定義する方法

✅ UnifyWeaver が各パターンをどのように検出・最適化するか

✅ 各パターンのパフォーマンス特性

✅ 各パターンをいつ使用すべきか

## 次のステップ

高度なコード解析と視覚化について学ぶために、**ノートブック 3: コールグラフの視覚化** へ進みましょう！